In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

import shap

from src.data_loader import load_data, basic_cleaning

# Task 4: Statistical Modeling & Risk-Based Pricing

The objective of this notebook is to develop predictive models that estimate insurance claim severity and support risk-based premium pricing.

The modeling workflow includes:

- Data preparation
- Feature engineering
- Regression modeling
- Model evaluation
- Model interpretability using SHAP
- Risk-based premium estimation

In [ ]:
DATA_PATH = "../data/insurance_data_cleaned.csv"

df = load_data(DATA_PATH)
df = basic_cleaning(df)

df.head()

In [ ]:
claims_df = df[df["TotalClaims"] > 0].copy()

claims_df.shape

In [ ]:
features = [
    "Province",
    "PostalCode",
    "VehicleType",
    "AutoMake",
    "VehicleModel",
    "RegistrationYear",
    "CustomValueEstimate",
    "Cylinders",
    "Kilowatts",
    "CapitalOutstanding",
    "NumberOfDoors",
    "Margin"
]

target = "TotalClaims"

In [ ]:
X = claims_df[features]
y = claims_df[target]

In [ ]:
categorical_features = X.select_dtypes(include="object").columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features, numerical_features

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

linear_preds = linear_model.predict(X_test)

linear_rmse = np.sqrt(
    mean_squared_error(y_test, linear_preds)
)

linear_r2 = r2_score(y_test, linear_preds)

print("Linear Regression RMSE:", linear_rmse)
print("Linear Regression R²:", linear_r2)

In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_preds)
)

rf_r2 = r2_score(y_test, rf_preds)

print("Random Forest RMSE:", rf_rmse)
print("Random Forest R²:", rf_r2)

In [ ]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42
        ))
    ]
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(
    mean_squared_error(y_test, xgb_preds)
)

xgb_r2 = r2_score(y_test, xgb_preds)

print("XGBoost RMSE:", xgb_rmse)
print("XGBoost R²:", xgb_r2)

In [ ]:
model_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "RMSE": [
        linear_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R²": [
        linear_r2,
        rf_r2,
        xgb_r2
    ]
})

model_results

In [ ]:
best_model_name = model_results.sort_values(
    "RMSE"
).iloc[0]["Model"]

print("Best Model:", best_model_name)

In [ ]:
X_processed = preprocessor.fit_transform(X_train)

xgb_only = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_only.fit(X_processed, y_train)

explainer = shap.Explainer(xgb_only)

shap_values = explainer(X_processed[:1000])

shap.plots.beeswarm(shap_values)

# SHAP Business Interpretation

The SHAP analysis identifies the most influential variables driving claim severity predictions.

Key findings may include:

- Higher vehicle value contributes to larger predicted claims.
- Geographic location influences insurance risk exposure.
- Vehicle age and vehicle type significantly affect expected claim severity.
- Certain vehicle brands/models are associated with elevated claim risk.

These insights support ACIS in developing more precise and explainable pricing policies.

In [ ]:
claims_df["PredictedSeverity"] = xgb_model.predict(X)

expense_loading = 500
profit_margin = 0.15

claims_df["SuggestedPremium"] = (
    claims_df["PredictedSeverity"]
    + expense_loading
)

claims_df["SuggestedPremium"] = (
    claims_df["SuggestedPremium"] *
    (1 + profit_margin)
)

claims_df[
    [
        "TotalPremium",
        "PredictedSeverity",
        "SuggestedPremium"
    ]
].head()

# Final Recommendations

## Pricing Strategy

- Introduce province-specific pricing adjustments for high-risk geographic areas.
- Increase premiums for high-risk vehicle categories identified by the predictive models.
- Offer reduced premiums to low-risk customer segments to improve market competitiveness.

## Marketing Strategy

- Prioritize customer acquisition in profitable geographic regions.
- Develop retention campaigns for low-risk vehicle owners.

## Model Deployment

- Deploy the XGBoost model as the primary severity prediction engine.
- Use SHAP interpretability outputs to support underwriting transparency and regulatory compliance.

## Future Improvements

- Incorporate driver behavioral data if available.
- Add real-time telematics features.
- Explore deep learning approaches for more advanced risk modeling.